# 01 – Data Exploration

This notebook gives an overview of the **GTSRB** (German Traffic Sign Recognition Benchmark) dataset:
- class distribution
- sample images per class
- image-size statistics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

sns.set_style('whitegrid')
DATA_ROOT = Path('../data')
print('PyTorch:', torch.__version__)

## 1. Load dataset

In [ ]:
# torchvision downloads the dataset automatically on first run
basic_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

train_dataset = datasets.GTSRB(
    root=DATA_ROOT, split='train', download=True, transform=basic_transform
)
test_dataset = datasets.GTSRB(
    root=DATA_ROOT, split='test', download=True, transform=basic_transform
)

print(f'Training samples : {len(train_dataset):,}')
print(f'Test samples     : {len(test_dataset):,}')
print(f'Number of classes: {len(train_dataset.classes)}')

## 2. Class names

In [ ]:
CLASS_NAMES = [
    'Speed limit 20',    'Speed limit 30',    'Speed limit 50',
    'Speed limit 60',    'Speed limit 70',    'Speed limit 80',
    'End speed limit 80','Speed limit 100',   'Speed limit 120',
    'No passing',        'No passing >3.5t',  'Right-of-way',
    'Priority road',     'Yield',             'Stop',
    'No vehicles',       'No vehicles >3.5t', 'No entry',
    'General caution',   'Dangerous curve L', 'Dangerous curve R',
    'Double curve',      'Bumpy road',        'Slippery road',
    'Road narrows R',    'Road work',         'Traffic signals',
    'Pedestrians',       'Children crossing', 'Bicycles crossing',
    'Ice/snow',          'Wild animals',      'End all limits',
    'Turn right ahead',  'Turn left ahead',   'Go straight',
    'Go straight or R',  'Go straight or L',  'Keep right',
    'Keep left',         'Roundabout',        'End no passing',
    'End no passing >3.5t'
]
print(f'Class names defined: {len(CLASS_NAMES)}')

## 3. Class distribution

In [ ]:
train_labels = [label for _, label in train_dataset._samples]
class_counts = pd.Series(train_labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(class_counts.index, class_counts.values, color='steelblue')
ax.set_xlabel('Class ID')
ax.set_ylabel('Number of training images')
ax.set_title('GTSRB – Training set class distribution')
ax.set_xticks(class_counts.index)
ax.set_xticklabels(class_counts.index, rotation=90)
plt.tight_layout()
plt.show()

print(f'Min samples per class: {class_counts.min():,}  (class {class_counts.idxmin()})')
print(f'Max samples per class: {class_counts.max():,}  (class {class_counts.idxmax()})')

## 4. Sample images per class

In [ ]:
# Display one example image for each of the 43 classes
fig, axes = plt.subplots(5, 9, figsize=(18, 10))
axes = axes.flatten()

# Build a map: class_id -> first sample index
class_to_idx = {}
for idx, (_, label) in enumerate(train_dataset._samples):
    if label not in class_to_idx:
        class_to_idx[label] = idx
    if len(class_to_idx) == 43:
        break

for class_id in range(43):
    img_tensor, _ = train_dataset[class_to_idx[class_id]]
    img = img_tensor.permute(1, 2, 0).numpy()
    axes[class_id].imshow(img)
    axes[class_id].set_title(f'{class_id}', fontsize=8)
    axes[class_id].axis('off')

for ax in axes[43:]:
    ax.axis('off')

plt.suptitle('GTSRB – One sample per class', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Image-size statistics

In [ ]:
# Collect original image sizes (before resize transform)
raw_dataset = datasets.GTSRB(
    root=DATA_ROOT, split='train', download=False, transform=transforms.ToTensor()
)

widths, heights = [], []
sample_size = min(2000, len(raw_dataset))
indices = np.random.choice(len(raw_dataset), sample_size, replace=False)

for i in indices:
    img_path = raw_dataset._samples[i][0]
    with Image.open(img_path) as img:
        w, h = img.size
    widths.append(w)
    heights.append(h)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color='steelblue', edgecolor='white')
ax1.set_title('Image width distribution')
ax1.set_xlabel('Width (px)')

ax2.hist(heights, bins=30, color='salmon', edgecolor='white')
ax2.set_title('Image height distribution')
ax2.set_xlabel('Height (px)')

plt.tight_layout()
plt.show()

print(f'Width  – mean: {np.mean(widths):.1f} px, median: {np.median(widths):.1f} px')
print(f'Height – mean: {np.mean(heights):.1f} px, median: {np.median(heights):.1f} px')